In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 量子ポートフォリオオプティマイザー: Global Data QuantumによるQiskit Function


> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premiumプラン、Flexプラン、およびOn-Prem（IBM Quantum Platform API経由）プランのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される場合があります。
# 概要
量子ポートフォリオオプティマイザーは、動的ポートフォリオ最適化問題に取り組むQiskit Functionです。これは、リターンを最大化しリスクを最小化するために、一連の資産にわたる定期的な投資のリバランスを目的とする、金融における標準的な問題です。最先端の量子最適化技術を活用することで、この関数はプロセスを簡素化し、量子コンピューティングの専門知識がないユーザーでも、最適な投資トラジェクトリーを見つけるうえでその利点を活用できるようにします。ポートフォリオマネージャー、定量的金融の研究者、および個人投資家に最適で、このツールはポートフォリオ最適化における取引戦略のバックテストを可能にします。
# 関数の説明
量子ポートフォリオオプティマイザー関数は、変分量子固有値ソルバー（VQE）アルゴリズムを使用して二次制約なし二値最適化（QUBO）問題を解き、動的ポートフォリオ最適化問題に対処します。ユーザーは資産価格データを提供し、投資制約を定義するだけで、関数が量子最適化プロセスを実行し、最適化された投資トラジェクトリーのセットを返します。

このプロセスは4つの主要なステージで構成されています。まず、入力データを量子互換の問題にマッピングし、動的ポートフォリオ最適化問題のQUBOを構築し、量子演算子（イジングハミルトニアン）に変換します。次に、入力問題とVQEアルゴリズムを量子ハードウェア上で実行できるように適応させます。その後、VQEアルゴリズムを量子ハードウェア上で実行し、最後に結果をポストプロセッシングして最適な投資トラジェクトリーを提供します。このシステムにはノイズ対応（[SQD](/guides/qiskit-addons-sqd)ベース）のポストプロセッシングも含まれており、出力の品質を最大化します。

このQiskit FunctionはGlobal Data Quantumによる[公開論文](https://arxiv.org/abs/2412.19150)に基づいています。
![関数のワークフローの可視化](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# 入力
関数の入力引数を以下の表に説明します。資産データとその他の問題仕様は必須です。また、最適化プロセスをカスタマイズするためにVQEの設定を含めることもできます。
| **名前**               | **型**             | **説明**                                                       | **必須** | **デフォルト**                       | **例**                                   |
|------------------------|--------------------|----------------------------------------------------------------|----------|--------------------------------------|------------------------------------------|
| assets                 | json               | 資産価格を含む辞書                                             | はい     | -                                    | -                                        |
| qubo_settings          | json               | QUBOの設定                                                     | はい     | -                                    | 以下の表の例を参照                       |
| ansatz_settings        | json               | アンザッツの設定                                               | いいえ   | `None`                                | 以下の表の例を参照                       |
| optimizer_settings     | json               | オプティマイザーの設定                                         | いいえ   | `None`                                | 以下の表の例を参照                       |
| backend                | str                | QPUバックエンド名                                              | いいえ   |  -    | `"ibm_torino"`                           |
| previous_session_id    | strのリスト        | 以前の実行からデータを取得するためのセッションIDのリスト[(*)](#1-note) | いいえ   | 空リスト                             | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | ノイズ対応SQDポストプロセッシングを適用する                    | いいえ   | `True`                                | `True`                                   |
| tags                   | 文字列のリスト     | 実験を識別するためのタグのリスト                               | いいえ   | 空リスト                             | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*実行を再開する場合や、1つ以上の以前のセッションで処理されたジョブを取得する場合は、セッションIDのリストを`previous_session_id`パラメーターに渡す必要があります。これは、プロセスのエラーにより最適化タスクが完了しなかった場合や、実行を完了させる必要がある場合に特に便利です。これを実現するには、最初の実行で使用したものと同じ引数を提供し、説明に従って`previous_session_id`リストを追加してください。
> **Caution:** 以前のセッションのデータの読み込み（最適化を再開するため）には最大1時間かかる場合があります。
## `assets`
データは、特定の日付における金融資産の終値に関する情報を格納するJSONオブジェクトとして構造化する必要があります。形式は以下の通りです：

- プライマリキー（文字列）：金融資産の名前またはティッカーシンボル（例：「8801.T」）。
- セカンダリキー（文字列）：YYYY-MM-DD形式の日付。
- 値（数値）：指定された日付における資産の終値。価格は正規化されていてもされていなくても入力できます。

*すべての辞書が同じセカンダリキー（日付）を持つ必要があることに注意してください。特定の資産に他の資産が持つ日付がない場合は、一貫性を確保するためにデータを補完する必要があります。例えば、その資産の直近の終値を使用することで対応できます。*
### 例
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** 資産データには、少なくとも``(nt+1) * dt``（[`qubo_settings`](#qubo_settings)入力セクションを参照）タイムスタンプ（例：日数）の終値が含まれている必要があります。
## `qubo_settings`
次の表は`qubo_settings`辞書のキーを説明しています。タイムステップ数`nt`、分解量子ビット数`nq`、および`max_investment`を指定して辞書を構築するか、その他のデフォルト値を変更してください。
| 名前                | 型      | 説明                                                                         | 必須     | デフォルト | 例      |
|---------------------|---------|------------------------------------------------------------------------------|----------|------------|---------|
| nt                  | int     | タイムステップ数                                                             | はい     | -          | 4       |
| nq                  | int     | 分解量子ビット数                                                             | はい     | -          | 4       |
| max_investment      | float   | すべての資産にわたる投資通貨単位の最大数                                     | はい     | -          | 10      |
| dt*                  | int     | 各タイムステップで考慮される時間ウィンドウ。単位は資産データのキー間の時間間隔と一致します | いいえ   | 30         | -       |
| risk_aversion       | float     | リスク回避係数                                                              | いいえ   | 1000       | -       |
| transaction_fee     | float     | 取引手数料係数                                                              | いいえ   | 0.01       | -       |
| restriction_coeff   | float   | QUBO定式化内で問題の制約を強制するために使用されるラグランジュ乗数           | いいえ   | 1          | -       |
## `ansatz_settings`
デフォルトオプションを変更するには、以下のキーを持つ`ansatz_settings`パラメーターの辞書を作成してください。デフォルトでは、アンザッツは`"real_amplitudes"`に設定されており、両方の追加オプション（以下の表を参照）は`False`に設定されています。
| 名前                  | 型       | 説明                                                                        | 必須     | デフォルト |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|------------|
| ansatz[*](#single-star)                | str      | 使用するアンザッツ                                          | いいえ   | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | 複数のパスマネージャーサブルーチンを有効にします（Tailoredアンザッツでは利用不可） | いいえ   | `False`   |
| dd_enable   | bool     | ダイナミカルデカップリングを追加します                        | いいえ   | `False`   |

<span id="single-star"></span>
\* 利用可能なアンザッツ
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored`（`ibm_torino`バックエンド、7資産、4タイムステップ、4分解量子ビットのみ）

<span id="double-star"></span>
** ``multiple_passmanager``が``False``に設定されている場合、関数はデフォルトのQiskitパスマネージャー（`optimization_level=3`）を使用します。``True``に設定されている場合、``multiple_passmanager``サブルーチンは3つのパスマネージャーを比較します：前述のデフォルトQiskitパスマネージャー、QPUの最近傍チェーン上に量子ビットをマッピングするパスマネージャー、および[AIトランスパイラー](/guides/ai-transpiler-passes)サービス。その後、推定される累積エラーが最も低いパスマネージャーが選択されます。
## `optimizer_settings`
このパラメーターは、最適化プロセスの調整可能なオプションを持つ辞書です。
| 名前               | 型     | 説明                                         | 必須     | デフォルト               |
|--------------------|--------|----------------------------------------------|----------|--------------------------|
| primitive_options  | json   | プリミティブの設定                            | いいえ   | -                        |
| optimizer          | str    | 選択されたクラシカルオプティマイザー          | いいえ   | `"differential_evolution"` |
| optimizer_options  | json   | オプティマイザーの設定                        | いいえ   | -                        |
> **Note:** 現在、利用可能なオプティマイザーオプションは`"differential_evolution"`のみです。

`primitive_options`および`optimizer_options`キーの下に、以下のパラメーターを持つ辞書を設定します：
### `primitive_options`
| 名前              | 型     | 説明                                       | 必須     | デフォルト                  | 例                         |
|-------------------|--------|--------------------------------------------|----------|-----------------------------|----------------------------|
| sampler_shots     | int    | Samplerのショット数                        | いいえ   | 100000                      | -                          |
| estimator_shots   | int    | Estimatorのショット数                      | いいえ   | 25000                       | -                          |
| estimator_precision | float | 期待値の所望精度。指定した場合、`estimator_shots`の代わりに精度が使用されます。 | いいえ | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int または str | ランタイムセッションが強制終了される前に開いたままにできる最大時間。秒（int）または`"2h 30m 40s"`のような文字列で指定できます。システムが定める最大値未満である必要があります。 | いいえ | `None` | `"1h 15m"`                |
### `optimizer_options`
| 名前              | 型       | 説明                                     | 必須     | デフォルト    |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | 世代数                                   | いいえ   | 20            |
| population_size   | int      | 集団サイズ                               | いいえ   | 20            |
| mutation_range    | list     | 最大および最小変異係数                   | いいえ   | [0, 0.25]     |
| recombination     | float    | 組換え係数                               | いいえ   | 0.4           |
| max_parallel_jobs | int      | 並列実行されるQPUジョブの最大数          | いいえ   | 3             |
| max_batchsize | int      | 最大バッチサイズ                         | いいえ   | 200           |
> **Note:** - 差分進化によって評価される世代数は、初期集団が含まれるため`num_generations` + 1となります。
> - 回路の総数は`(num_generations + 1) * population_size`として計算されます。
> - 一般的に、集団サイズを大きくし世代数を増やすと、最適化結果の品質が向上します。ただし、集団サイズを120を超えて設定し、世代数を20より大きくすること（例：``120 * 21 = 2520``回路の合計）は推奨されません。これにより回路数が過多となり、計算コストと処理時間が大幅に増加します。
> 
> - この関数は以前の最適化を再開することができ、世代数を増やすことは常に可能です（`previous_session_id`以外は同じ入力を提供し、``num_generations``を増加させることで対応できます）。
> **Note:** Qiskit Runtimeのジョブ制限に準拠していることを確認してください。
> 
> - Sampler：`sampler_shots <= 10_000_000`。
> - Estimator：`max_batchsize * estimator_shots * observable_size <= 10_000_000`（この関数では、オブザーバブルのすべての項が可換であるため`observable_size=1`）。
> 
> 詳細は[ジョブ制限](/guides/job-limits)ガイドを参照してください。
# 出力
関数は2つの辞書を返します：最適な解とその関連する最小目的コストを含む最良の最適化結果を格納する`"result"`辞書と、最適化プロセス中に得られたすべての結果のデータとその各メトリクスを含む`"metadata"`です。

最初の辞書は最良のパフォーマンスを示す解に焦点を当て、2番目の辞書は目的コストやその他の関連メトリクスを含む、すべての解に関する詳細情報を提供します。

## 出力辞書：
| **名前**    | **型**                       | **説明**                                                                        | **例**       |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | 時系列の投資戦略を格納しており、各タイムスタンプが資産固有の投資ウェイト（各ウェイトは総投資額で正規化された投資額）にマッピングされています。 | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | 解、コスト、メトリクスを含む分析中に生成されたデータ。                          | 以下の例を参照 |
### `metadata`辞書の説明
| **名前**                 | **型**                | **説明**                                                                                            | **例**                        |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|-------------------------------|
| session_id               | str                   | IBM Quantumセッションの一意の識別子。                                     | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | コストや制約など、各ポストプロセッシングされたサンプルの各種メトリクスを含む辞書。 | 以下の説明を参照              |
| sampler_counts           | dict[str, int]        | キーがサンプリングされた解のビット文字列表現で、値がそのカウントである辞書。 | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | 投資戦略内の各タイムステップにおける資産の投資順序を示すリスト。 | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | 問題のQUBO行列。                                                                                    | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | プロセスの各ステージにわたるCPUおよびQPUの使用時間（秒）の概要。                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### `all_samples_metrics`辞書の説明
| **名前**                | **型**         | **説明**                                                                                                         | **例**                        |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|-------------------------------|
| investment_trajectories | list[list]     | デコードされた量子状態から導出された投資戦略。 | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | 各投資トラジェクトリーがサンプリングされた回数。インデックスは`investment_trajectories`と対応します。 | `[5, 3]`                     |
| objective_costs         | list[float]    | 各投資トラジェクトリーの目的関数値。最小から最大の順に並べられています。                             | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | 各投資トラジェクトリーのリスク調整後パフォーマンス（シャープ比）。インデックス順に整列されています。 | `[1.1, 0.7]`                 |
| returns                 | list[float]    | 各投資トラジェクトリーの期待リターン。インデックス順に整列されています。                             | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | 各投資トラジェクトリー内の最大制約逸脱量。インデックス順に整列されています。                        | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | 各投資トラジェクトリーに関連する推定取引コスト。インデックス順に整列されています。                  | `[0.01, 0.02]`               |
# はじめに
[APIキー](https://quantum.cloud.ibm.com/)を使用して認証し、以下のようにQiskit Functionを選択してください。（このコードスニペットは、[アカウントをローカル環境に保存済み](/guides/functions#install-qiskit-functions-catalog-client)であることを前提としています。）

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## 例：7資産を用いた動的ポートフォリオ最適化
この例では、動的ポートフォリオ最適化（DPO）関数を実行し、最適なパフォーマンスを得るためにその設定を調整する方法を示します。最適な結果を得るためにパラメーターをファインチューニングするための詳細な手順が含まれています。

このケースでは、7資産、4タイムステップ、4分解量子ビットを使用しており、合計112量子ビットが必要となります。
### 1. ポートフォリオに含まれる資産を読み込む。
ポートフォリオのすべての資産が特定のパスのフォルダーに保存されている場合、以下の関数を使用して`pandas.DataFrame`に読み込み、`dict`形式のオブジェクトに変換できます。

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

この例では、[8801.T](https://finance.yahoo.com/quote/8801.T)、[CLF](https://finance.yahoo.com/quote/CLF)、[GBPJPY](https://finance.yahoo.com/quote/GBPJPY)、[ITX.MC](https://finance.yahoo.com/quote/ITX.MC)、[META](https://finance.yahoo.com/quote/META)、[TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y)、および[XS2239553048](https://finance.yahoo.com/quote/XS2239553048)の資産を使用しています。以下の図は、この例で使用したデータを示しており、2023年1月1日から9月1日までの資産の日次終値の推移を示しています。

この例では、日付全体で均一性を確保するために、非取引日には直前の利用可能な日付の終値を補完しています。選択した資産が取引日の異なる複数の市場に属しているため、データセットの一貫性を標準化するためにこのステップを適用しています。
![資産の過去データの可視化](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. 問題を定義する。

`qubo_settings`辞書でパラメーターを設定して、問題の仕様を定義してください。

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. オプティマイザーとアンザッツの設定を定義する。（オプション）
オプションで、オプティマイザーとそのパラメーターの選択、およびプリミティブとその設定の指定を含む、最適化プロセスに関する特定の要件を定義します。

Tailored Ansatzの場合、選択された集団サイズは、この値が安定した効率的な最適化をもたらすことを示す以前の実験に基づいています。

Real Amplitudes Ansatzの場合、``population_size``と回路内の量子ビット数の間の線形関係に従うことができます。経験則として、`real_amplitudes`アンザッツには**最低**``population_size ~ 0.8 * n_qubits``を使用することが推奨されます。

Optimized Real AmplitudesはReal Amplitudesアンザッツよりも優れた最適化性能を発揮することが期待されます。ただし、このアンザッツで最適化する変数の数はReal Amplitudesの場合よりもはるかに速く増加します（[論文](https://arxiv.org/pdf/2412.19150)を参照）。したがって、大規模な問題では、Optimized Real Amplitudesはより多くの回路実行が必要です。Optimized Real Amplitudesは最大100量子ビットを必要とする問題に有効である可能性が高いですが、``population_size``パラメーターを設定する際には注意が必要です。この``population_size``のスケールアップの例として、前の表では84量子ビットの問題でOptimize Real Amplitudesが120の``population_size``を必要とするのに対し、56量子ビットの問題では40の``population_size``で十分であることを示しています。

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

特定のアンザッツを選択することもできます。以下では``'Tailored'``アンザッツを使用します。

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. 問題を実行する。

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. 結果を取得する。
[出力](#output)セクションで述べたように、関数は目的関数値に基づいて最小から最大の順に並べられた投資トラジェクトリーを含む辞書を返します。この結果セットにより、最低コストのトラジェクトリーとその対応する投資評価を特定できます。また、さまざまなトラジェクトリーの分析が可能で、特定のニーズや目標に最も合致するものを選択しやすくなります。この柔軟性により、さまざまな好みやシナリオに合わせた選択が可能です。
まず、プロセス中に見つかった最低の目的コストを達成した結果戦略を提示します。

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>